# Synthetic graph detection and event tracking

Generate a new run with `python -m reddit_sna synthetic-week1 --config configs/detection_development.toml --run-id week1-synthetic-demo` first. This notebook reads verified saved outputs; it does not tune thresholds or rerun the detector. The recipes were already inspected, so these are descriptive synthetic results, not a new unseen experiment. See `docs/RUNBOOK.md` and `reports/WEEK1_REPORT.md`.

For another run, set `REDDIT_SYNTHETIC_REFERENCE` to its `pair-reference.json`. Legacy T02 replay remains possible with `configs/legacy-handoff-reference.json` plus explicit bundle and tracking paths. No cached output is committed in this source notebook.

## Environment and package APIs

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML, JSON
import matplotlib.pyplot as plt
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
ROOT = Path(os.environ.get("REDDIT_SNA_PROJECT", Path.cwd())).resolve()
if "REDDIT_SNA_PROJECT" not in os.environ:
    ROOT = next((p for p in [ROOT, *ROOT.parents] if (p / "pyproject.toml").is_file()), ROOT)
if not (ROOT / "pyproject.toml").is_file():
    raise ValueError("Set REDDIT_SNA_PROJECT to the repository root")
print("Active interpreter:", sys.executable)
print("Repository:", ROOT)

from reddit_sna.reporting.synthetic_demo import load_synthetic_demo, build_synthetic_figures, export_synthetic_demo, table_html
from reddit_sna.reporting.synthetic_cases import analyze_synthetic_cases, case_summary, build_case_figures, export_case_analysis
from reddit_sna.reporting.tracking_demo import load_tracking_demo, build_tracking_figures, export_tracking_demo

## Explicit input reference and fresh output paths

In [ ]:
REFERENCE = Path(os.environ.get("REDDIT_SYNTHETIC_REFERENCE", ROOT / "artifacts/detection-tracking/week1-synthetic-demo/pair-reference.json")).resolve()
reference = json.loads(REFERENCE.read_text(encoding="utf-8"))
BUNDLE = Path(os.environ.get("REDDIT_SYNTHETIC_BUNDLE", reference.get("bundle", ROOT / "artifacts/handoff/phuc-dang-week1-20261019-001"))).resolve()
DATASET = os.environ.get("REDDIT_SYNTHETIC_DEMO_DATASET", "dev-grow-easy-g11")
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = Path(os.environ.get("REDDIT_SYNTHETIC_DEMO_OUTPUT", ROOT / "artifacts/detection-tracking" / f"synthetic-notebook-{STAMP}")).resolve()
ANALYSIS_OUTPUT = OUTPUT.with_name(OUTPUT.name + "-cases")
TRACKING_OUTPUT = OUTPUT.with_name(OUTPUT.name + "-tracking")
NOTEBOOK_SOURCE = ROOT / "notebooks/detection_tracking/01_synthetic_demo.ipynb"
for destination in (OUTPUT, ANALYSIS_OUTPUT, TRACKING_OUTPUT):
    if destination.exists() or not destination.is_relative_to(ROOT / "artifacts") or destination.is_relative_to(BUNDLE) or BUNDLE.is_relative_to(destination):
        raise ValueError("Choose a new output inside artifacts, separate from inputs")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print("Bundle:", BUNDLE)
print("Interpretation:", reference.get("interpretation", "Historical fixed T02 replay; original chronology is not recertified"))

## Verify all source runs before reading scores

In [ ]:
demo = load_synthetic_demo(BUNDLE, expected_delivery_sha256=reference["delivery_manifest_sha256"], dataset_id=DATASET)
display(HTML(table_html([dict(split=s, **i["counts"], source_manifest_sha256=i["source_manifest_sha256"]) for s,i in demo["inventories"].items()])))
display(JSON(demo["recipes"]["development"]))

## Detection quality, coverage and undefined AMI

In [ ]:
summary = [r for r in demo["summary_view"] if r["level"] == "all_scenarios"]
display(HTML(table_html(summary)))
print("Missing AMI is retained with its reason; coverage 1.0 does not imply correct partitions.")

## Full selected networks: display does not determine metrics

In [ ]:
figures = build_synthetic_figures(demo)
for figure in figures.values():
    plt.close(figure)
display(figures["01_network"])
display(HTML(table_html([s["metric"] for s in demo["network"]["snapshots"]])))

## Canonical quality and diagnostic seed variation

In [ ]:
display(figures["02_quality"])
display(figures["03_seeds"])
print("Seed 0 remains canonical; diagnostic seeds are not independent test datasets.")

## Recorded stage timing

In [ ]:
display(HTML(table_html(demo["timings"])))
display(figures["04_timing"])
print("Peak native memory was not measured; no zero value is invented.")

## Post-hoc difficult and undefined-score cases

In [ ]:
cases = analyze_synthetic_cases(demo, split="development")
display(HTML(table_html(case_summary(cases))))
case_figures = build_case_figures(cases)
for figure in case_figures.values():
    plt.close(figure)
    display(figure)

## Explain the errors with retained graph evidence

In [ ]:
for case in cases:
    details = case["description"]
    print(case["dataset_id"], case["month"], "truth groups:", details["truth_groups"], "detected groups:", details["predicted_groups"])
    display(HTML(table_html(details["edge_blocks"])))
    display(JSON(case["inputs"]))
print("These selected examples are explanatory diagnostics, not a causal proof or parameter selection rule.")

## Export detection and case analysis with provenance

In [ ]:
metadata = export_synthetic_demo(demo, figures, OUTPUT, notebook=NOTEBOOK_SOURCE)
analysis_metadata = export_case_analysis(demo, cases, case_figures, ANALYSIS_OUTPUT, notebook=NOTEBOOK_SOURCE)
print("Detection export:", OUTPUT, metadata["status"])
print("Case export:", ANALYSIS_OUTPUT, analysis_metadata["status"])

## Truth tracking versus detector-plus-tracking

In [ ]:
legacy_pair = ROOT / "artifacts/detection-tracking/phuc-dang-final-20261020-001"
paths = reference.get("tracking", {})
TRACKING_DEVELOPMENT = Path(os.environ.get("REDDIT_TRACKING_DEVELOPMENT", paths.get("development", legacy_pair / "development"))).resolve()
TRACKING_EVALUATION = Path(os.environ.get("REDDIT_TRACKING_EVALUATION", paths.get("evaluation", legacy_pair / "evaluation"))).resolve()
tracking = load_tracking_demo(TRACKING_DEVELOPMENT, TRACKING_EVALUATION, input_pins=reference.get("source_manifest_sha256"))
display(HTML(table_html(tracking["summary"])))
print("Mean dataset F1 and pooled F1 are different summaries. Preserve unmatched truth and extra detector groups.")

## Confusion matrices retain all six events

In [ ]:
tracking_figures = build_tracking_figures(tracking)
for figure in tracking_figures.values():
    plt.close(figure)
    display(figure)

## Per-class metrics and actual failures

In [ ]:
display(HTML(table_html(tracking["per_class"])))
display(HTML(table_html(tracking["errors"], ["split", "dataset_id", "source_community_id", "truth_event", "predicted_event"])))
print("Source mapping is an evaluator operation; truth does not influence the tracker.")

## Save tracking results and state the limits

In [ ]:
tracking_metadata = export_tracking_demo(tracking, tracking_figures, TRACKING_OUTPUT)
print("Tracking export:", TRACKING_OUTPUT)
display(JSON(tracking_metadata))
print("Synthetic benchmark only. Historical partner acceptance, original Dang freeze, unseen scientific evaluation and paper novelty are not certified by this notebook.")